# Google Play Store Analysis – Task 2

## Objective
To analyze the relationship between installs, ratings, and estimated revenue across app categories, and understand performance differences between Free and Paid apps.

## Analysis Goal
To identify which app categories achieve the best balance between user reach (installs) and monetization (revenue), and to understand whether popularity translates into profitability.


# Google Play Store Analysis – Task 2

## Objective
To analyze the relationship between installs, ratings, and estimated revenue across app categories, and understand performance differences between Free and Paid apps.

## Dataset
Google Play Store dataset containing app-level information such as installs, ratings, price, category, and size.

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import pytz

pd.set_option('display.max_columns', None)
%matplotlib inline

print("Libraries imported successfully.")

---
## Step 2 — Load Dataset

In [ ]:
df = pd.read_csv('playstore_data.csv')

print(f"Shape: {df.shape}")
df.head()

---
## Step 3 — Data Cleaning

In [ ]:
# Robust Size Cleaning (handles M, k, and Varies)
def clean_size(size):
    if size == 'Varies with device' or pd.isna(size):
        return None
    if 'M' in size:
        return float(size.replace('M',''))
    if 'k' in size:
        return float(size.replace('k',''))/1024
    return None

df['Size'] = df['Size'].apply(clean_size)
df = df.dropna(subset=['Size'])


---
## Step 4 — Revenue Estimation

Revenue is approximated as:

**Revenue = Price × Installs**

This assumes all installs convert into purchases for paid apps.
While simplified, it helps compare monetization potential across categories.

In [ ]:
# Revenue Estimation (simplified)
# Assumption: Revenue = Price × Installs
df['Revenue_Est'] = df['Price'] * df['Installs']


---
## Step 5 — Apply Filters

Filters applied to retain only meaningful, comparable apps:
- **Installs ≥ 10,000** — removes niche apps with very low reach
- **Size > 15 MB** — focuses on feature-rich apps
- **Android Ver contains '4'** — ensures broad modern device compatibility
- **Content Rating = Everyone** — consistent target audience
- **Revenue ≥ $10,000** (Paid apps only) — commercially viable paid apps only

In [ ]:
df = df[df['Android Ver'].notna()]

---
## Step 6 — Identify Top 3 Categories

In [ ]:
top3_categories = (
    filtered_df
    .groupby('Category')['Installs']
    .sum()
    .nlargest(3)
    .index
    .tolist()
)

print(f"Top 3 Categories: {top3_categories}")

df_top3 = filtered_df[filtered_df['Category'].isin(top3_categories)]

---
## Step 7 — Time Gate (1 PM – 2 PM IST)

> This chart is restricted to **1:00 PM – 2:00 PM IST** as per dashboard requirements.  
> Outside this window, the chart is suppressed and a notice is displayed.

In [ ]:
def is_within_ist_window(start_hour=13, end_hour=14):
    """Returns True only if current IST time is within [start_hour, end_hour)."""
    ist = pytz.timezone('Asia/Kolkata')
    now_ist = datetime.now(ist)
    print(f"Current IST time: {now_ist.strftime('%I:%M %p')}")
    return start_hour <= now_ist.hour < end_hour

CHART_ALLOWED = is_within_ist_window()

---
## Step 8 — Dual-Axis Chart: Installs vs Revenue (Free vs Paid)

- **Primary Y-axis (bars):** Average Installs  
- **Secondary Y-axis (line):** Average Estimated Revenue  
- Grouped by **Free** and **Paid** within each of the **Top 3 Categories**

In [ ]:
import matplotlib.pyplot as plt

grouped = filtered_df.groupby('Category').agg({
    'Installs': 'mean',
    'Revenue_Est': 'mean'
}).sort_values(by='Installs', ascending=False).head(5)

fig, ax1 = plt.subplots(figsize=(10,5))

ax1.bar(grouped.index, grouped['Installs'])
ax1.set_ylabel('Average Installs')

ax2 = ax1.twinx()
ax2.plot(grouped.index, grouped['Revenue_Est'], marker='o')
ax2.set_ylabel('Estimated Revenue')

plt.title('Top Categories: Installs vs Revenue')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


---
## Key Insight

Free apps dominate in terms of installs, indicating higher user reach.

However, paid apps generate higher revenue per user, suggesting that a **freemium model** (free app with paid features) could maximize both reach and monetization.

Additionally, high installs do not always correlate with high revenue, highlighting the importance of **pricing strategy**.

---
## Conclusion

- App installs are strongly influenced by accessibility — free apps perform significantly better in reach.
- Revenue is driven more by pricing strategy than installs alone.
- Categories vary significantly in performance, indicating niche-specific strategies are important.
- A balanced monetization model (freemium) is likely the most effective approach for developers aiming to maximize both growth and earnings.

---
*Task 2 Complete — Google Play Store Analysis Project*